[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MahkameSalimi/ISPRS-Tutorial/blob/main/intro_quantum_and_lstm/qiskit_intro_tutorial.ipynb)

# Hands-on Introduction to Qubits, Bloch Spheres, Gates, and Measurements with Qiskit

**Goal:** by the end of this notebook, learners should be able to:

1. Describe a single qubit as a state vector.
2. Visualize one-qubit states on the Bloch sphere.
3. Apply simple gates and rotations and see how they move the state.
4. Measure a circuit on a simulator and interpret the counts.
5. Build and run a slightly more complicated circuit.

This notebook uses **Qiskit** for circuits and visualization.

## 0. Setup

Run this cell first. If Qiskit is not installed, uncomment the installation line.

> In Google Colab or a fresh environment, you may need:
> ```python
> !pip install qiskit qiskit-aer matplotlib pylatexenc ipywidgets
> ```

In [ ]:
# !pip install qiskit qiskit-aer qiskit-ibm-runtime matplotlib pylatexenc ipywidgets

import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_bloch_multivector, plot_histogram

# Optional simulator backend from qiskit-aer.
# The notebook also includes a pure-Statevector fallback for measurement sampling.
try:
    from qiskit_aer import AerSimulator
    HAS_AER = True
except Exception:
    HAS_AER = False

print("Aer available:", HAS_AER)

## 1. What is a qubit?

A classical bit is either `0` or `1`.

A qubit can be in a linear combination of two basis states:

$$
|\psi\rangle = \alpha |0\rangle + \beta |1\rangle,
$$

where $\alpha$ and $\beta$ are complex numbers called **amplitudes**.

The measurement probabilities are:

$$
P(0) = |\alpha|^2, \qquad P(1) = |\beta|^2.
$$

Because total probability must be 1:

$$
|\alpha|^2 + |\beta|^2 = 1.
$$

For one qubit, we can visualize the state as a point on the **Bloch sphere**.

### 1.1 The state $|0\rangle$

In Qiskit, a fresh qubit starts in state $|0\rangle$.

In [ ]:
qc = QuantumCircuit(1)
qc.draw("mpl")

In [ ]:
state = Statevector.from_instruction(qc)
print(state)
plot_bloch_multivector(state)

On the Bloch sphere, $|0\rangle$ points upward along the positive $z$ axis.

### 1.2 The state $|1\rangle$

The `X` gate flips $|0\rangle$ to $|1\rangle$.

You can think of `X` as the quantum version of a classical NOT gate.

In [ ]:
qc = QuantumCircuit(1)
qc.x(0)

state = Statevector.from_instruction(qc)
print(state)
qc.draw("mpl")

In [ ]:
plot_bloch_multivector(state)

$|1\rangle$ points downward along the negative $z$ axis.

## 2. Superposition with the Hadamard gate

The Hadamard gate `H` creates a balanced superposition:

$$
H|0\rangle = |+\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}.
$$

If we measure $|+\rangle$, we expect about 50% zeros and 50% ones.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)

state = Statevector.from_instruction(qc)
print(state)
qc.draw("mpl")

In [ ]:
plot_bloch_multivector(state)

On the Bloch sphere, $|+\rangle$ points along the positive $x$ axis.

The qubit is not secretly either `0` or `1`. Before measurement, it is in a quantum state with amplitudes for both outcomes.

## 3. Rotations on the Bloch sphere

Single-qubit rotation gates move the Bloch vector around the sphere.

Qiskit has three common rotation gates:

$$
R_x(\theta), \qquad R_y(\theta), \qquad R_z(\theta).
$$

They rotate the qubit state around the $x$, $y$, and $z$ axes.

### 3.1 Rotation around the $y$ axis

Start from $|0\rangle$. Apply $R_y(\theta)$ for different angles.

In [ ]:
def show_ry(theta):
    qc = QuantumCircuit(1)
    qc.ry(theta, 0)
    state = Statevector.from_instruction(qc)
    print(f"theta = {theta:.3f} rad = {theta/np.pi:.2f}π")
    print("Statevector:", state)
    display(qc.draw("mpl"))
    return plot_bloch_multivector(state)

show_ry(np.pi/4)

In [ ]:
# Try several angles.
for theta in [0, np.pi/4, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]:
    qc = QuantumCircuit(1)
    qc.ry(theta, 0)
    state = Statevector.from_instruction(qc)
    print(f"theta = {theta/np.pi:.2f}π")
    display(plot_bloch_multivector(state))
    plt.show()

### 3.2 Optional interactive Bloch sphere

Run this cell if your notebook environment supports `ipywidgets`.

Move the slider and watch how $R_y(\theta)$ changes the qubit.

In [ ]:
try:
    from ipywidgets import interact, FloatSlider

    @interact(theta=FloatSlider(value=0, min=0, max=2*np.pi, step=0.05, description='theta'))
    def interactive_ry(theta):
        qc = QuantumCircuit(1)
        qc.ry(theta, 0)
        state = Statevector.from_instruction(qc)
        display(qc.draw("mpl"))
        display(plot_bloch_multivector(state))
except Exception as e:
    print("ipywidgets is not available in this environment.")
    print(e)
    

### 3.3 Compare $R_x$, $R_y$, and $R_z$

Important intuition:

- $R_x$ rotates around the $x$ axis.
- $R_y$ rotates around the $y$ axis.
- $R_z$ rotates around the $z$ axis.

Notice that $R_z$ applied to $|0\rangle$ does not visibly move the Bloch vector because $|0\rangle$ is already on the $z$ axis. To see the effect of $R_z$, first create $|+\rangle$ using `H`.

In [ ]:
def compare_rotations(theta=np.pi/2):
    circuits = {}

    qc_x = QuantumCircuit(1)
    qc_x.rx(theta, 0)
    circuits["Rx(theta) on |0>"] = qc_x

    qc_y = QuantumCircuit(1)
    qc_y.ry(theta, 0)
    circuits["Ry(theta) on |0>"] = qc_y

    qc_z = QuantumCircuit(1)
    qc_z.h(0)
    qc_z.rz(theta, 0)
    circuits["H then Rz(theta)"] = qc_z

    for title, qc in circuits.items():
        state = Statevector.from_instruction(qc)
        print("\n", title)
        display(qc.draw("mpl"))
        display(plot_bloch_multivector(state))
        plt.show()

compare_rotations(np.pi/2)

## 4. Common one-qubit gates

Here are some important gates:

| Gate | Main idea | Effect on $\|0\rangle$ |
|---|---|---|
| `X` | bit flip | $\|0\rangle \to \|1\rangle$ |
| `H` | creates superposition | $\|0\rangle \to \|+\rangle$ |
| `Z` | phase flip | no visible change on $\|0\rangle$ |
| `S` | phase rotation by $\pi/2$ | no visible change on $\|0\rangle$ |
| `T` | phase rotation by $\pi/4$ | no visible change on $\|0\rangle$ |

Phase gates are easiest to see after creating a superposition first.

In [ ]:
# Compare H, then phase gates.
gates = {
    "H only: |+>": lambda qc: qc.h(0),
    "H then Z": lambda qc: (qc.h(0), qc.z(0)),
    "H then S": lambda qc: (qc.h(0), qc.s(0)),
    "H then T": lambda qc: (qc.h(0), qc.t(0)),
}

for name, build in gates.items():
    qc = QuantumCircuit(1)
    build(qc)
    state = Statevector.from_instruction(qc)
    print("\n" + name)
    display(qc.draw("mpl"))
    display(plot_bloch_multivector(state))
    plt.show()

## 5. Measurement on a simulator

When we measure, we do not directly see amplitudes. We see classical outcomes such as `0` or `1`.

If the state is

$$
|\psi\rangle = \alpha |0\rangle + \beta |1\rangle,
$$

then after many shots:

- outcome `0` appears about $|\alpha|^2$ of the time,
- outcome `1` appears about $|\beta|^2$ of the time.

A **shot** means one repeated run of the same circuit.

### 5.1 Helper function for running measurements

This helper first tries to use `AerSimulator`. If Aer is not installed, it samples from the exact Statevector probabilities.

In [ ]:
def run_counts(qc_without_measurement, shots=1024, seed=123):
    """Run a circuit and return measurement counts.

    Input circuit should NOT contain measurement.
    The function adds measurement to all qubits.
    """
    qc = qc_without_measurement.copy()
    qc.measure_all()

    if HAS_AER:
        simulator = AerSimulator(seed_simulator=seed)
        result = simulator.run(qc, shots=shots).result()
        return result.get_counts()

    # Fallback: exact probabilities from Statevector, then random sampling.
    state = Statevector.from_instruction(qc_without_measurement)
    probs = state.probabilities_dict()

    rng = np.random.default_rng(seed)
    outcomes = list(probs.keys())
    pvals = np.array([probs[o] for o in outcomes], dtype=float)
    samples = rng.choice(outcomes, size=shots, p=pvals)

    counts = {o: int(np.sum(samples == o)) for o in outcomes}
    return counts

### 5.2 Measure $|0\rangle$

We should get `0` almost every time.

In [ ]:
qc = QuantumCircuit(1)
counts = run_counts(qc, shots=1024)
print(counts)
plot_histogram(counts)

### 5.3 Measure $|1\rangle$

Apply `X`, then measure. We should get `1` almost every time.

In [ ]:
qc = QuantumCircuit(1)
qc.x(0)
counts = run_counts(qc, shots=1024)
print(counts)
plot_histogram(counts)

### 5.4 Measure $|+\rangle$

Apply `H`, then measure. We should get approximately 50% `0` and 50% `1`.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
counts = run_counts(qc, shots=1024)
print(counts)
plot_histogram(counts)

### 5.5 Measure after a rotation

For the circuit below:

$$
R_y(\theta)|0\rangle
$$

we expect:

$$
P(0) = \cos^2\left(\frac{\theta}{2}\right), \qquad
P(1) = \sin^2\left(\frac{\theta}{2}\right).
$$

Let us compare theory and simulation.

In [ ]:
theta = np.pi / 3

qc = QuantumCircuit(1)
qc.ry(theta, 0)

counts = run_counts(qc, shots=4096)

p0_theory = np.cos(theta/2)**2
p1_theory = np.sin(theta/2)**2

print("Theoretical probabilities:")
print("P(0) =", p0_theory)
print("P(1) =", p1_theory)

print("\nSimulator counts:")
print(counts)

display(qc.draw("mpl"))
display(plot_bloch_multivector(Statevector.from_instruction(qc)))
plot_histogram(counts)

## 6. Two-qubit circuits and entanglement

Now we introduce two important gates:

1. `H` on qubit 0 creates superposition.
2. `CX` uses qubit 0 as the control and qubit 1 as the target.

The circuit below creates a Bell state:

$$
|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}.
$$

This is an entangled state. If you measure many times, you should see mostly `00` and `11`, not `01` and `10`.

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)

state = Statevector.from_instruction(bell)
print(state)
bell.draw("mpl")

In [ ]:
counts = run_counts(bell, shots=2048)
print(counts)
plot_histogram(counts)

### Important note about Bloch spheres for entangled states

A single Bloch sphere works best for a single isolated qubit.

For entangled states, the full two-qubit state cannot be fully represented by two independent Bloch vectors. Qiskit may show reduced one-qubit Bloch spheres, but they do not contain all the information about the entangled state.

In [ ]:
plot_bloch_multivector(state)

## 7. A more complicated circuit: three-qubit GHZ state

The GHZ state is like a larger Bell state:

$$
|GHZ\rangle = \frac{|000\rangle + |111\rangle}{\sqrt{2}}.
$$

The circuit is:

1. Apply `H` to qubit 0.
2. Apply `CX(0, 1)`.
3. Apply `CX(1, 2)`.

After measurement, we expect mostly `000` and `111`.

In [ ]:
ghz = QuantumCircuit(3)
ghz.h(0)
ghz.cx(0, 1)
ghz.cx(1, 2)

display(ghz.draw("mpl"))

counts = run_counts(ghz, shots=4096)
print(counts)
plot_histogram(counts)

## 8. Run the Bell state on IBM Quantum hardware

In this section we send a small Bell-state circuit to a real IBM Quantum
backend.

* If you do not have access to IBM Quantum, the cell automatically falls
  back to the local `AerSimulator`.
* If the connection or the job submission fails for any reason, the cell
  falls back to the local simulator.
* If the queue or the execution exceeds `HARDWARE_TIMEOUT_SECONDS`, the
  job is cancelled and the cell falls back to the local simulator.
* Nighthawk-family processors are excluded from automatic backend
  selection (per IBM guidance for this tutorial). You can adjust the
  `EXCLUDED_BACKEND_KEYWORDS` list below if needed.

### What you need before running

1. An IBM Quantum account (for example via PINQ).
2. An API token exported as the environment variable `IBM_QUANTUM_TOKEN`
   (or already saved through `QiskitRuntimeService.save_account(...)`).
3. The `qiskit-ibm-runtime` package installed in your notebook environment.

If any of these are missing, the cell will still run and produce
simulator results so that the rest of the tutorial continues to work.


In [ ]:
# Uncomment if the package is not already installed:
# !pip install qiskit-ibm-runtime

import os
import time

from qiskit import QuantumCircuit, transpile
from qiskit.visualization import plot_histogram

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
SHOTS = 1024
# Maximum total time (in seconds) we are willing to wait for the hardware
# job to finish (queue + execution). After this, we cancel and fall back.
HARDWARE_TIMEOUT_SECONDS = 600  # 10 minutes
# Backends whose name contains any of these keywords are skipped during
# automatic selection. Nighthawk processors are excluded per IBM guidance.
EXCLUDED_BACKEND_KEYWORDS = ("nighthawk",)


# ---------------------------------------------------------------------------
# Bell-state circuit (same for hardware and the simulator fallback)
# ---------------------------------------------------------------------------
bell_hw = QuantumCircuit(2, 2)
bell_hw.h(0)
bell_hw.cx(0, 1)
bell_hw.measure([0, 1], [0, 1])
display(bell_hw.draw("mpl"))


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def _run_local_fallback(circuit, shots, reason):
    """Run the circuit on a local AerSimulator and return (counts, label)."""
    print(f"\n[Fallback] {reason}")
    print("[Fallback] Running on the local AerSimulator instead.\n")
    from qiskit_aer import AerSimulator

    sim = AerSimulator()
    tqc = transpile(circuit, sim)
    sim_job = sim.run(tqc, shots=shots)
    return sim_job.result().get_counts(), "AerSimulator (local fallback)"


def _extract_counts(sampler_result):
    """Extract counts from a SamplerV2 result without hardcoding the
    classical-register name."""
    data = sampler_result[0].data
    # DataBin is iterable over its field names in recent qiskit-ibm-runtime.
    for name in list(data):
        field = getattr(data, name)
        if hasattr(field, "get_counts"):
            return field.get_counts()
    raise RuntimeError("Could not find a BitArray in the sampler result.")


# ---------------------------------------------------------------------------
# Try hardware, fall back to a local simulator on any failure
# ---------------------------------------------------------------------------
counts_hw = None
backend_label = None

try:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

    token = os.getenv("IBM_QUANTUM_TOKEN")
    if token:
        # Save (or refresh) the account using the token from the environment.
        QiskitRuntimeService.save_account(
            channel="ibm_quantum",
            token=token,
            overwrite=True,
        )

    service = QiskitRuntimeService(channel="ibm_quantum")

    # Filter out Nighthawk-family backends (and anything else listed in
    # EXCLUDED_BACKEND_KEYWORDS), then pick the least busy one.
    candidates = [
        b
        for b in service.backends(simulator=False, operational=True, min_num_qubits=2)
        if not any(kw in b.name.lower() for kw in EXCLUDED_BACKEND_KEYWORDS)
    ]
    if not candidates:
        raise RuntimeError("No eligible IBM backends available after exclusions.")

    backend = min(candidates, key=lambda b: b.status().pending_jobs)
    print(
        f"Selected backend: {backend.name} "
        f"(pending jobs: {backend.status().pending_jobs})"
    )

    transpiled_bell = transpile(bell_hw, backend=backend, optimization_level=1)
    sampler = Sampler(mode=backend)
    job = sampler.run([transpiled_bell], shots=SHOTS)
    print(f"Submitted job: {job.job_id()}")
    print(f"Waiting up to {HARDWARE_TIMEOUT_SECONDS}s for the result...")

    # Poll for completion so we can enforce a timeout on queue + execution.
    start = time.time()
    terminal_states = {"DONE", "CANCELLED", "ERROR"}
    while True:
        status = str(job.status()).split(".")[-1]
        if status in terminal_states:
            break
        if time.time() - start > HARDWARE_TIMEOUT_SECONDS:
            try:
                job.cancel()
            except Exception:
                pass
            raise TimeoutError(
                f"Hardware job exceeded {HARDWARE_TIMEOUT_SECONDS}s "
                "(queue + execution); cancelled."
            )
        time.sleep(5)

    if status != "DONE":
        raise RuntimeError(f"Hardware job ended in state {status}.")

    counts_hw = _extract_counts(job.result())
    backend_label = backend.name

except Exception as exc:
    counts_hw, backend_label = _run_local_fallback(
        bell_hw,
        SHOTS,
        f"Could not obtain results from IBM Quantum hardware: {exc}",
    )


print(f"\nResults from: {backend_label}")
print("Counts:", counts_hw)
plot_histogram(counts_hw)


## 9. Exercises

Try these exercises during the tutorial. Some have hints, but learners should first try without looking at the solution.

### Exercise 1: Create $|1\rangle$

Create a one-qubit circuit that prepares $|1\rangle$ from $|0\rangle$.

Then:

1. Draw the circuit.
2. Plot the Bloch sphere.
3. Measure it for 1024 shots.

**Expected result:** almost all counts should be `1`.

In [ ]:
# Exercise 1: write your code here.

qc = QuantumCircuit(1)
# TODO: add a gate here

state = Statevector.from_instruction(qc)
display(qc.draw("mpl"))
display(plot_bloch_multivector(state))

counts = run_counts(qc, shots=1024)
print(counts)
plot_histogram(counts)

### Exercise 2: Create an equal superposition

Create $|+\rangle = (|0\rangle + |1\rangle)/\sqrt{2}$.

Then measure it for 2048 shots.

**Expected result:** around half `0` and half `1`.

In [ ]:
# Exercise 2: write your code here.

qc = QuantumCircuit(1)
# TODO: add a gate here

state = Statevector.from_instruction(qc)
display(qc.draw("mpl"))
display(plot_bloch_multivector(state))

counts = run_counts(qc, shots=2048)
print(counts)
plot_histogram(counts)

### Exercise 3: Find a rotation angle

Use `ry(theta, 0)` to make the probability of measuring `1` close to 25%.

Hint:

$$
P(1) = \sin^2(\theta/2).
$$

Try different values of `theta`.

In [ ]:
# Exercise 3: change theta until P(1) is close to 0.25.

theta = None  # TODO: replace None with a number, for example np.pi/3

qc = QuantumCircuit(1)
if theta is not None:
    qc.ry(theta, 0)

state = Statevector.from_instruction(qc)
display(qc.draw("mpl"))
display(plot_bloch_multivector(state))

counts = run_counts(qc, shots=4096)
print(counts)
plot_histogram(counts)

### Exercise 4: Bell state

Build the Bell-state circuit yourself:

$$
\frac{|00\rangle + |11\rangle}{\sqrt{2}}.
$$

Steps:

1. Create a two-qubit circuit.
2. Apply `H` to qubit 0.
3. Apply `CX` from qubit 0 to qubit 1.
4. Measure the circuit.

**Expected result:** mostly `00` and `11`.

In [ ]:
# Exercise 4: write your code here.

qc = QuantumCircuit(2)
# TODO: add gates here

state = Statevector.from_instruction(qc)
display(qc.draw("mpl"))

counts = run_counts(qc, shots=2048)
print(counts)
plot_histogram(counts)

### Exercise 5: Make your own three-qubit circuit

Build a three-qubit circuit using at least:

- one `H` gate,
- one rotation gate such as `rx`, `ry`, or `rz`,
- one `CX` gate.

Then:

1. Draw the circuit.
2. Print the final statevector.
3. Measure it.
4. Explain why the histogram looks the way it does.

In [ ]:
# Exercise 5: open-ended circuit.

qc = QuantumCircuit(3)
# TODO: create your own circuit
# Example ideas:
# qc.h(0)
# qc.ry(np.pi/3, 1)
# qc.cx(0, 2)

state = Statevector.from_instruction(qc)
print(state)
display(qc.draw("mpl"))

counts = run_counts(qc, shots=4096)
print(counts)
plot_histogram(counts)

## 9. Solutions

Open this section after learners try the exercises.

### Solution 1

In [ ]:
qc = QuantumCircuit(1)
qc.x(0)

state = Statevector.from_instruction(qc)
display(qc.draw("mpl"))
display(plot_bloch_multivector(state))

counts = run_counts(qc, shots=1024)
print(counts)
plot_histogram(counts)

### Solution 2

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)

state = Statevector.from_instruction(qc)
display(qc.draw("mpl"))
display(plot_bloch_multivector(state))

counts = run_counts(qc, shots=2048)
print(counts)
plot_histogram(counts)

### Solution 3

We need:

$$
\sin^2(\theta/2) = 0.25.
$$

So:

$$
\sin(\theta/2) = 0.5,
$$

which gives:

$$
\theta/2 = \pi/6, \qquad \theta = \pi/3.
$$

In [ ]:
theta = np.pi / 3

qc = QuantumCircuit(1)
qc.ry(theta, 0)

state = Statevector.from_instruction(qc)
display(qc.draw("mpl"))
display(plot_bloch_multivector(state))

counts = run_counts(qc, shots=4096)
print(counts)
plot_histogram(counts)

### Solution 4

In [ ]:
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

state = Statevector.from_instruction(qc)
print(state)
display(qc.draw("mpl"))

counts = run_counts(qc, shots=2048)
print(counts)
plot_histogram(counts)

### Solution 5: one possible answer

In [ ]:
qc = QuantumCircuit(3)
qc.h(0)
qc.ry(np.pi/3, 1)
qc.cx(0, 2)
qc.cx(1, 2)

state = Statevector.from_instruction(qc)
print(state)
display(qc.draw("mpl"))

counts = run_counts(qc, shots=4096)
print(counts)
plot_histogram(counts)

## 12. Bridge toward LSTM and QLSTM

The first part of this notebook teaches the **language of quantum circuits**: qubits, gates, rotations, measurement, and entanglement.

Before learners study **Quantum LSTM (QLSTM)**, they also need one more layer of ideas:

1. how classical data becomes quantum circuit inputs,
2. how trainable parameters are placed inside a quantum circuit,
3. how a circuit produces numerical outputs,
4. how a classical optimizer updates the circuit parameters,
5. how this becomes a quantum layer inside a neural network.

This is the bridge from **basic circuits** to **variational quantum circuits (VQCs)** and then to **QLSTM**.

### 12.1 Classical data encoding

In machine learning, the input is usually a vector, for example

$$
x = [x_1, x_2, x_3].
$$

A quantum circuit cannot directly accept this vector like a normal Python function. We must encode the numbers into gates.

A simple method is **angle encoding**:

$$
x_i \longrightarrow R_y(x_i).
$$

So each input feature becomes a rotation angle.

In [ ]:
# Example: angle encoding for a small classical vector

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_bloch_multivector
import numpy as np

x = [0.2, 1.0, 2.0]  # three classical features

qc = QuantumCircuit(3)
for i, value in enumerate(x):
    qc.ry(value, i)

state = Statevector.from_instruction(qc)
display(qc.draw("mpl"))
display(plot_bloch_multivector(state))

**Important idea:** the data changes the quantum state. Different input vectors create different points on the Bloch spheres.

### 12.2 Trainable quantum circuits

A variational quantum circuit has two types of angles:

- **input angles**: come from the data,
- **trainable angles**: learned during training.

This is similar to a neural network layer. In a neural network, we train weights. In a VQC, we train gate angles.

In [ ]:
from qiskit.circuit import ParameterVector

num_qubits = 3
x_params = ParameterVector("x", num_qubits)
theta = ParameterVector("theta", num_qubits)

qc = QuantumCircuit(num_qubits)

# Data encoding layer
for i in range(num_qubits):
    qc.ry(x_params[i], i)

# Trainable layer
for i in range(num_qubits):
    qc.rx(theta[i], i)

# Entangling layer
qc.cx(0, 1)
qc.cx(1, 2)

display(qc.draw("mpl"))

This circuit is not yet numerical because it contains symbols: `x[0]`, `theta[0]`, etc. We must assign numbers before simulation.

In [ ]:
# Assign numerical values to the input and trainable parameters

x_values = [0.2, 1.0, 2.0]
theta_values = [0.5, -0.3, 0.8]

parameter_values = {}
for i in range(num_qubits):
    parameter_values[x_params[i]] = x_values[i]
    parameter_values[theta[i]] = theta_values[i]

bound_qc = qc.assign_parameters(parameter_values)
state = Statevector.from_instruction(bound_qc)

display(bound_qc.draw("mpl"))
print(state)

### 12.3 Circuit output as expectation value

For quantum machine learning, we usually do not only look at bitstrings like `010` or `111`.

Instead, we often measure an **expectation value**, for example:

$$
\langle Z_0 \rangle.
$$

This value is between $-1$ and $+1$:

- close to $+1$: qubit is more like $|0\rangle$,
- close to $-1$: qubit is more like $|1\rangle$,
- close to $0$: balanced between $|0\rangle$ and $|1\rangle$.

This gives us a normal numerical output that can be passed to a classical neural network.

In [ ]:
from qiskit.quantum_info import SparsePauliOp

# Measure Z on qubit 0 in a 3-qubit system.
# Qiskit Pauli strings are written with the rightmost character acting on qubit 0.
observable = SparsePauliOp.from_list([("IIZ", 1.0)])

expectation = state.expectation_value(observable).real
print("<Z0> =", expectation)

### 12.4 A tiny quantum layer

Now we can write a small function that behaves like a quantum layer:

$$
\text{quantum\_layer}(x, \theta) \rightarrow \text{number}.
$$

This is the key idea behind many hybrid quantum-classical models.

In [ ]:
def quantum_layer(x_values, theta_values):
    """Return one expectation value from a small variational quantum circuit."""
    num_qubits = len(x_values)
    qc = QuantumCircuit(num_qubits)

    # Encode data
    for i, value in enumerate(x_values):
        qc.ry(value, i)

    # Trainable rotations
    for i, value in enumerate(theta_values):
        qc.rx(value, i)

    # Entanglement
    for i in range(num_qubits - 1):
        qc.cx(i, i + 1)

    state = Statevector.from_instruction(qc)

    # Z on qubit 0: rightmost Pauli acts on qubit 0
    pauli_string = "I" * (num_qubits - 1) + "Z"
    observable = SparsePauliOp.from_list([(pauli_string, 1.0)])

    return state.expectation_value(observable).real

x_sample = [0.2, 1.0, 2.0]
theta_sample = [0.5, -0.3, 0.8]

print(quantum_layer(x_sample, theta_sample))

### 12.5 Why this matters for QLSTM

A classical LSTM has gates such as:

- forget gate,
- input gate,
- candidate/update gate,
- output gate.

Each gate is normally computed using classical neural-network layers.

In a QLSTM, the main idea is to replace some of those classical layers with **variational quantum circuits**.

So instead of only doing something like:

$$
f_t = \sigma(W_f [h_{t-1}, x_t] + b_f),
$$

we may use a quantum circuit as part of the gate computation:

$$
f_t = \sigma(\text{ClassicalPostProcess}(\text{VQC}_f([h_{t-1}, x_t]))).
$$

The VQC acts like a trainable feature map.

### 12.6 Extra exercises before QLSTM

**Exercise A:** Change the data vector `x = [0.2, 1.0, 2.0]`. How does the expectation value change?

**Exercise B:** Add another trainable layer after the entangling layer. Does the output become more flexible?

**Exercise C:** Return three expectation values instead of one: $\langle Z_0\rangle$, $\langle Z_1\rangle$, and $\langle Z_2\rangle$.

**Exercise D:** Build four separate VQCs and name them like LSTM gates: `forget_vqc`, `input_vqc`, `candidate_vqc`, and `output_vqc`.

**Exercise E:** Write a paragraph explaining why a QLSTM is still a hybrid model, not a fully quantum neural network.

### 12.7 Suggested answers for the extra exercises

**Exercise A:** The expectation value changes because the input vector controls the data-encoding rotations. A different input creates a different quantum state.

**Exercise B:** Yes, usually adding more trainable layers increases expressivity. But too many layers can make training harder and may increase noise on real quantum hardware.

**Exercise C:** Use Pauli strings `IIZ`, `IZI`, and `ZII` for qubits 0, 1, and 2 respectively.

**Exercise D:** The four VQCs can have the same structure but independent trainable parameters. This is similar to classical LSTM gates having different weight matrices.

**Exercise E:** A QLSTM is hybrid because the data, loss function, optimizer, and often part of the neural network are classical. The quantum circuit acts as a trainable sub-layer inside the larger classical model.

## 13. Extra questions

1. Why does $|+\rangle$ give random-looking measurement results even though the state preparation is deterministic?
2. Why does applying `Rz` to $|0\rangle$ not visibly change the Bloch sphere?
3. What is the difference between a statevector and measurement counts?
4. Why can a Bell state not be fully explained by two separate one-qubit Bloch spheres?
5. What changes if we increase the number of shots from 100 to 10,000?

